# eClipseBord — EDA: NASA Five Millennium Eclipse Catalog

Målet här är att:

1. förstå datasetets struktur
2. få en känsla för fördelningen av förmörkelsetyper och tidsspann
3. bestämma vilka kolumner som är relevanta att visa i dashboarden

Dataset: NASA:s "Five Millennium Catalog" av sol- och månförmörkelser
(-1999 till år 3000), nedladdat från Kaggle. Två filer: solar.csv och
lunar.csv.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## 1. Ladda in data

In [ ]:
solar = pd.read_csv("../data/solar.csv")
lunar = pd.read_csv("../data/lunar.csv")

print("Solar shape:", solar.shape)
print("Lunar shape:", lunar.shape)

solar.head()

## 2. Struktur och datatyper

Kollar dtypes och de första raderna. Notera: Latitude/Longitude är
strängar med kompassriktning (till exempel "6.0N"), inte rena float-värden —
det behöver vi tänka på om vi vill plotta koordinater i dashboarden.

In [ ]:
solar.dtypes


In [ ]:
lunar.dtypes


## 3. Saknade värden

Path Width och Central Duration saknas för solförmörkelser som
inte har en central linje och det är förväntat, inte ett datafel.

I lunar.csv markeras "ej tillämpligt" med strängen "-" istället för
riktig NaN, vi räknar dem separat.


In [ ]:
print("Saknade värden (solar):")
print(solar.isna().sum())


In [ ]:
print("Antal '-' i lunar-kolumner: ")
for col in ["Partial Eclipse Duration (m)", "Total Eclipse Duration (m)"]:
    print(f"  {col}: {(lunar[col] == '-').sum()} av {len(lunar)}")


## 4. Fördelning av förmörkelsetyper

Solförmörkelser klassificeras till exempel som Total (T), Annular (A), Partial (P)
och Hybrid (H), med varianter som också kan vara (A+) eller (Tm) för specialfall, att förmörkelsen är nära polerna.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
solar["Eclipse Type"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Antal solförmörkelser per typ")
ax.set_xlabel("Typ")
ax.set_ylabel("Antal")
plt.tight_layout()
plt.show()


In [ ]:
lunar["Eclipse Type"].value_counts()


## 5. Tidsspann

Datumen är strängar som "-1999 June 12" alltså, år sedan månad/dag. Kalendern går ända från 2000 f.Kr. till år 3000. Vi extraherar året för att kunna gruppera och plotta över tid. 


In [ ]:
def extract_year(date_str: str) -> int:
    """Plockar ut året från en sträng som '-1999 June 12' eller '2026 August 12'."""
    return int(date_str.split()[0])

solar["Year"] = solar["Calendar Date"].apply(extract_year)
lunar["Year"] = lunar["Calendar Date"].apply(extract_year)

solar["Year"].min(), solar["Year"].max()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
solar.groupby((solar["Year"] // 100) * 100).size().plot(ax=ax)
ax.set_title("Antal solförmörkelser per sekel")
ax.set_xlabel("Sekel (startår)")
ax.set_ylabel("Antal")
plt.tight_layout()
plt.show()


## 6. Numeriska kolumner - grundläggande statistik

Gamma beskriver hur centralt månens skugga passerar jorden, och Eclipse Magnitude hur stor andel av solen som täcks. 


In [ ]:
solar[["Gamma", "Eclipse Magnitude", "Sun Altitude", "Sun Azimuth"]].describe()


## 7. 2026 års förmörkelser

Sanity-check: caset nämner solförmörkelsen 12 augusti 2026 — den ska finnas i datan.


In [ ]:
solar[solar["Calendar Date"].str.contains("2026")]


## 8. Slutsatser inför dashboarden

- Datat är rent och komplett för kärnkolumnerna.
- Inget behov av tung data cleaning innan det används i backend/API.
- Latitude/Longitude behöver parsas om från kompassbokstav till tecken, för att vi ska kunna plotta förmörkelser på en karta.
- Path Width (km) / Central Duration (solar) och Total/Partial Eclipse Duration (lunar) har saknade värden "-" respektive "NaN" för icke-centrala/icke-totala förmörkelser.
- Relevanta kolumner exponera via FastAPI och visa i Streamlit: Calendar Date, Eclipse Type, Eclipse Magnitude/Umbral Magnitude, Latitude, Longitude, Path Width (km) / durations.
- Vi kan filtrera på år/typ i backend för att undvika att skicka alla ~12 000 rader till frontend på en gång.
